# Problem Set 8: Project Genesis – The Sovereign Sentinel

In the past, sensitive telemetry data, physical anomalies, and protected system states were transmitted over unsecured networks to external cloud servers. Every millisecond of latency threatened the stability of your real-time loops; every network failure led to the immediate collapse of your digital twin.

Today, we construct the **Fortress of Sovereignty**. You will embed a highly advanced, locally executable model from the **Google Gemma 4** family deep within the foundation of your own hardware. Your mission is to couple a discrete-event queue simulation—the beating heart of a virtual production line—with this local Sentinel. Without sending a single data packet to the outside world, the Sentinel will monitor telemetry, perform logical analysis in a protected thought channel, access a local knowledge base, and actively regulate the physics of the queue.

Prepare your terminals. Activate your local Tensor Cores. Data sovereignty is the key to absolute control. Let's build!

## Exercise 1: Initializing the Local Sentinel (Gemma 4 Edge)

We are using the highly efficient **Gemma 4 E2B-IT** variant. Despite its compact size (2.3 billion effective parameters), this model demonstrates excellent logical reasoning capabilities thanks to the novel Per-Layer Embeddings (PLE) architecture.

### Instructions:
1. Log into [Kaggle](https://www.kaggle.com) and accept the terms of use on the official Gemma 4 model page.
2. Create an API token in your Kaggle account settings to download the `kaggle.json` file.
3. Set the environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY` in your Python runtime.
4. Download the model using `kagglehub` and initialize the inference pipeline.

In [ ]:
import os
import kagglehub

# Enter your credentials here (or use a .env file)
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_api_key"

try:
    print("Requesting weights from the Kaggle interface...")
    # Downloading the instruction-tuned Gemma 4 E2B variant via kagglehub
    model_path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")
    print(f"Model successfully located in system cache at: {model_path}")
except Exception as e:
    print(f"Error downloading the model: {e}")
    print("Ensure that the model license has been accepted on Kaggle and API keys are correctly configured.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Initializing inference pipeline (bfloat16 for VRAM efficiency)...")
try:
    processor = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    print("Local Sentinel online and ready for inference.")
except NameError:
    print("Model path not found. Please run the cell above first.")

**Question:** Why is this memory footprint reduction (compact parameters/bfloat16) a critical factor when executing parallel physical simulations on the same hardware?

*Enter your answer here:*

## Exercise 2: Activating the Cognitive Core (Native Thought Mode)

This process is natively initiated by placing the `<|think|>` token at the very beginning of the system message. The model outputs its logical chain of thought between the `<|channel>thought\n` and `<channel|>` separators.

### Instructions:
1. Implement an inference function `generate_thoughtful_response` that formats the system prompt to activate the native thought mode.
2. Use Regular Expressions (RegEx) to isolate the thought channel (`thought`) from the final response.
3. Write a helper function that strips the thought blocks from the conversation history before the next user interaction occurs.

In [ ]:
import re

def generate_thoughtful_response(system_prompt, user_message, max_new_tokens=512):
    messages = [
        {"role": "system", "content": f"<|think|>\n{system_prompt}"},
        {"role": "user", "content": user_message}
    ]
    
    # Creating the chat template via the integrated processor
    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Conversion to input tensors
    inputs = processor(text=prompt_text, return_tensors="pt").to(model.device)
    
    # Inference without gradient calculation for performance optimization
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic output for reproducible system reactions
            eos_token_id=processor.eos_token_id
        )
    
    # Isolation of the generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded_output = processor.decode(generated_tokens, skip_special_tokens=False)
    
    # RegEx pattern to isolate the thought channel
    thought_pattern = r"<\|channel>thought\s*(.*?)\s*<channel|>"
    thought_match = re.search(thought_pattern, decoded_output, re.DOTALL)
    
    thought_content = thought_match.group(1).strip() if thought_match else "No explicit thought process recorded."
    
    # Cleaning the response for the final command code
    clean_response = re.sub(thought_pattern, "", decoded_output, flags=re.DOTALL).strip()
    clean_response = clean_response.replace("<turn|>", "").replace("<bos>", "").replace("<eos>", "").strip()
    
    return thought_content, clean_response

In [ ]:
def clean_history(messages):
    """
    Helper function: Removes thought blocks from history (State Management).
    This prevents cognitive overload and token growth during multi-turn chats.
    """
    cleaned_messages = []
    thought_pattern = r"<\|channel>thought\s*.*?\s*<channel|>"
    for msg in messages:
        if msg["role"] == "model":
            clean_content = re.sub(thought_pattern, "", msg["content"], flags=re.DOTALL).strip()
            cleaned_messages.append({"role": "model", "content": clean_content})
        else:
            cleaned_messages.append(msg)
    return cleaned_messages

## Exercise 3: The Discrete Queue Simulation (Foundry)

Parts arrive, wait in a buffer, and are processed by a machine. We simulate this behavior stochastically. Arrival times follow a Poisson process with rate $\lambda$ (Arrival Rate), while processing times at the station follow a Poisson distribution with rate $\mu$ (Service Rate).

### Instructions:
1. Write a simulation class `DiscreteFoundry` representing the state of a buffer.
2. If the arrival rate exceeds the processing rate (Instability), the buffer fills up.
3. Upon reaching maximum capacity, a buffer overflow occurs (data loss/scrap).
4. The `step()` method must update system variables and export a JSON string.

In [ ]:
import numpy as np
import json

class DiscreteFoundry:
    def __init__(self, arrival_rate=2.5, service_rate=1.0, capacity=15):
        self.arrival_rate = arrival_rate
        self.service_rate = service_rate
        self.capacity = capacity
        self.buffer = 0
        self.dropped = 0
        self.processed = 0
        
    def step(self):
        # Stochastic approximation
        arrivals = np.random.poisson(self.arrival_rate)
        services = np.random.poisson(self.service_rate)
        
        self.buffer += arrivals
        
        # Check for overflow
        if self.buffer > self.capacity:
            overflow = self.buffer - self.capacity
            self.dropped += overflow
            self.buffer = self.capacity
            
        # Processing
        processed_now = min(self.buffer, services)
        self.buffer -= processed_now
        self.processed += processed_now
        
        return {
            "queue_length": self.buffer,
            "max_capacity": self.capacity,
            "dropped_parts": self.dropped,
            "processed_parts": self.processed,
            "current_service_rate": self.service_rate
        }

In [ ]:
# Instantiating the production line
production_line = DiscreteFoundry()
print("Initial Tick:", json.dumps(production_line.step()))

## Exercise 4: Closed-Loop Control with a Cognitive Entity

The telemetry data is passed to the local Sentinel. It analyzes the situation offline within its thought channel, makes a cognitive decision, and dictates a new value for the service rate, which is immediately fed back into the simulation.

### Instructions:
1. Design a system prompt for Gemma 4 instructing the model to act as an industrial controller.
2. The model must evaluate the system state and adjust the service rate (Interval: $[0.5, 3.0]$).
3. Enforce a strict output format to programmatically parse the new numerical parameter.
4. Run the controlled simulation over 8 time steps.

In [ ]:
sentinel_system_prompt = """
You are the Sovereign Sentinel (AI Controller) of an industrial production line.
Your objective is to minimize buffer overflows (dropped_parts) while simultaneously saving energy.
Maximum interval for the Service Rate: 0.5 to 3.0.
Analyze the telemetry: If queue_length is high, increase the rate drastically. If low, decrease it.
EXCLUSIVELY output the new value on the very last line in the exact format:
NEW_SERVICE_RATE: <value>
"""

print("=== START OF CLOSED-LOOP CONTROL ===")
try:
    for tick in range(8):
        # 1. System Step
        telemetry_data = production_line.step()
        print(f"\n[Tick {tick+1}] Telemetry Input: {json.dumps(telemetry_data)}")
        
        # 2. Prompt Generation
        user_query = f"Status report: {json.dumps(telemetry_data)}\nCurrently set Service Rate: {production_line.service_rate}. Decision?"
        
        # 3. Local Inference
        thought_process, action_response = generate_thoughtful_response(sentinel_system_prompt, user_query, max_new_tokens=256)
        
        print(f"💭 [Thought Channel]: {thought_process[:200]}... [truncated]")
        print(f"🤖 [Model Decision]: {action_response}")
        
        # 4. Parsing the Decision
        match = re.search(r"NEW_SERVICE_RATE:\s*([0-9.]+)", action_response)
        if match:
            proposed_rate = float(match.group(1))
            # Physical safeguarding via system bounds
            validated_rate = max(0.5, min(3.0, proposed_rate))
            production_line.service_rate = validated_rate
            print(f">>> System Action: Service Rate updated to {validated_rate} for Step {tick+2}.")
        else:
            print("!!! Parsing Error: No valid adjustment found. Maintaining current parameter.")
except NameError:
    print("The model was not successfully loaded into memory in Cell 4.")

## Exercise 5: Cloud-Assisted Fine-Tuning via Colab CLI

To offload data- and compute-intensive model training without interrupting the local terminal workflow, we utilize the new **Google Colab CLI** (`google-colab-cli`). It allows us to provision remote GPUs in Google data centers, execute scripts, and download the training results (LoRA adapters) directly back into our local workspace using a single command sequence.

### Scenario:
We want to fine-tune a **Gemma 4 E2B** model via QLoRA so that it formats telemetry reports directly into stable industrial control codes.

### Instructions:
1. Install the official Google Colab CLI in your terminal environment.
2. Create a local Python script `finetune_run.py` that initializes Gemma 4 for LoRA fine-tuning using Keras.
3. Use the Colab CLI command sequence to request a remote system with a T4 GPU, install packages, execute the script, and transfer the fully trained weight adapters (`safetensors`) back to your local folder.

In [ ]:
# Optional installation of the CLI in your local terminal:
# pip install google-colab-cli

In [ ]:
finetune_script = """
import keras
import keras_hub
import os

# Utilizing JAX backend for XLA compiled high-speed training
os.environ["KERAS_BACKEND"] = "jax"

print("Loading Gemma 4 E2B for LoRA Fine-Tuning...")
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")

# 2. Activation of Low-Rank Adaptation (LoRA)
gemma_lm.backbone.enable_lora(rank=4)

# 3. Formatting dummy training data
training_data = [
    {"prompt": "Status: queue_length=14. Action?", "response": "NEW_SERVICE_RATE: 3.0"},
    {"prompt": "Status: queue_length=1. Action?", "response": "NEW_SERVICE_RATE: 0.5"}
]
prompts = [f"{item['prompt']} -> {item['response']}" for item in training_data]

# 4. Compiling the model with an optimized learning rate
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    weighted_metrics=["accuracy"]
)

# 5. Training for a representative epoch
gemma_lm.fit(prompts, epochs=1, batch_size=2)

# 6. Saving the trained adapter in VM memory
gemma_lm.save_to_preset("./gemma4_lora_adapter")
print("Fine-tuning completed successfully. Weights saved locally in the Colab VM.")
"""

with open("finetune_run.py", "w") as f:
    f.write(finetune_script.strip())
print("Local script 'finetune_run.py' created and ready for dispatch!")

The following command sequence illustrates the transparent code uplink in the Bash shell:

```bash
# 1. Allocate a new remote VM with a high-performance T4 GPU
colab new -s gemma-tuning --gpu T4

# 2. Install state-of-the-art ML libraries directly on the remote instance via 'uv'
colab install -s gemma-tuning keras-hub keras tensorflow torch

# 3. Execute your local Python script directly on the remote GPU (Transparent Code Uplink)
colab exec -s gemma-tuning -f finetune_run.py

# 4. Safely download the result (the finalized LoRA weight adapter)
colab download -s gemma-tuning /content/gemma4_lora_adapter ./local_gemma4_adapter

# 5. Terminate the remote session to conserve compute resources
colab stop -s gemma-tuning
```

**Question:** How does the interaction between the Colab CLI and the local edge model protect the intellectual sovereignty of your simulation data?

*Enter your answer here:*

## Exercise 6: Context Injection (Local RAG for SOPs)

The Sentinel must consult specific machine guidelines (Standard Operating Procedures, SOPs) to select the optimal mode. In this exercise, you will implement an offline-capable RAG system that extracts the appropriate regulatory guideline from a local document index based on the current queue length and feeds it as context into the Gemma 4 model.

### Instructions:
1. Create a list of SOPs as a local text index.
2. Implement a simple distance-based search function (e.g., keyword matching or Cosine Similarity over word frequencies) to filter the most relevant SOP document based on queue length.
3. Integrate this document as additional context into the prompt for Gemma 4.

In [ ]:
# Local RAG Index
SOP_DATABASE = {
    "SOP-101_NORMAL": "Queue < 5. Keep machine in Eco-Mode. Throttle rate to 1.0.",
    "SOP-202_WARNING": "Queue >= 5 and <= 10. Machine in Normal operation. Increase rate to 2.0.",
    "SOP-999_CRITICAL": "Queue > 10. EMERGENCY PROTOCOL. Force rate to 3.0 immediately!"
}

def retrieve_relevant_sop(queue_length):
    if queue_length > 10:
        return SOP_DATABASE["SOP-999_CRITICAL"]
    elif queue_length >= 5:
        return SOP_DATABASE["SOP-202_WARNING"]
    else:
        return SOP_DATABASE["SOP-101_NORMAL"]

# Test run of the local RAG system
test_queue = 12
retrieved_sop = retrieve_relevant_sop(test_queue)
print(f"Extracted SOP for queue length {test_queue}:\n-> {retrieved_sop}")

In [ ]:
# Demonstration of RAG-augmented prompt injection
augmented_prompt = f"{sentinel_system_prompt}\n\nCRITICAL REGULATION (SOP) FOR THIS STEP:\n{retrieved_sop}"
demo_query = f"Status report: Queue={test_queue}, Capacity=15. Current Service Rate: 1.0. Decision?"

try:
    t, a = generate_thoughtful_response(augmented_prompt, demo_query, max_new_tokens=256)
    print("\n=== RAG-AUGMENTED DECISION ===")
    print("Thoughts:\n", t)
    print("\nAction:\n", a)
except NameError:
    print("Please ensure the Gemma model was loaded in Cell 4.")

**Reflection Question:**
In what way does this approach simplify adjusting the system logic when factory regulations change?

*Enter your answer here:*